In [432]:
# 라이브러리
# !pip install torch torchvision torchaudio

In [433]:
import torch
import torch.nn as nn
import torch.optim as optim

In [434]:
# 데이터셋을 생성
# 독립 종속 -> 파이토치에서 독립, 종속의 데이터의 타입은 Tensor 로 구성
# 독립 변수 -> 2차원 데이터
# 종속 변수 -> 2차원 데이터 (sklearn 에서는 1차원)
X = torch.tensor( [[1.0], [2.0], [3.0], [4.0]] )
# 종속 변수는 독립 변수 데이터에서 2를 곱하고 1을 더해준 값으로 생성
Y = torch.tensor( [ [3.0], [5.0], [7.0], [9.0] ] )

In [435]:
type(X)

torch.Tensor

In [436]:
# 선형 회귀
class LinearReg(nn.Module):
    # 생성자 함수
    def __init__(self):
        # self: 클래스가 생성된 메모리의 주소
        # super() : 부모 클래스를 의미 -> 상속
        super( LinearReg, self ).__init__()  # 부모 클래스의 생성자 함수를 실행
        # Linear()에서는 첫번째 인자로는 입력 데이터 개수
        # 두번째 인자의 출력의 크기
        self.linear = nn.Linear(1, 1)
    
    def forward(self, x):
        return self.linear(x)

In [437]:
# 모델을 생성 -> 회귀 모델
# LinearReg 라는 클래스에서 생성자 함수에 매개변수가 self을 제외하고 존재 하지 않기 때문에
# 클래스 생성 시 인자 값을 넣지 않는다
model = LinearReg()

In [438]:
# 손실 함수
criterion = nn.MSELoss()
# 가중치의 갱신(경사 하강법) lr은 가중치의 변환 시 변환 비율
optimizer = optim.SGD(model.parameters(), lr = 0.01)

In [439]:
# 500번 방복 실행하면서 Loss를 확인
for epoch in range(500):
    # 순전파
    Y_pred = model(X)
    # 손실 함수
    loss = criterion(Y_pred, Y)
    # 역전파
    optimizer.zero_grad() # 기울기 초기화
    loss.backward() # 기울기 계산
    optimizer.step() # 파라미터를 업데이트

    # 50회마다 loss의 값을 출력
    if (epoch + 1) % 50 == 0:
        print(f'Epoch : [{epoch+1}, 500], Loss : {round(loss.item(), 4)}')

Epoch : [50, 500], Loss : 0.0288
Epoch : [100, 500], Loss : 0.0213
Epoch : [150, 500], Loss : 0.0158
Epoch : [200, 500], Loss : 0.0117
Epoch : [250, 500], Loss : 0.0087
Epoch : [300, 500], Loss : 0.0064
Epoch : [350, 500], Loss : 0.0048
Epoch : [400, 500], Loss : 0.0035
Epoch : [450, 500], Loss : 0.0026
Epoch : [500, 500], Loss : 0.0019


In [440]:
# 예측 값 확인
pred = model(X).detach()

for i in range(len(X)):
    print(f"독립 : {X[i].item()}, 종속 : {Y[i].item()}, 예측 : {pred[i].item()}")

독립 : 1.0, 종속 : 3.0, 예측 : 2.92911958694458
독립 : 2.0, 종속 : 5.0, 예측 : 4.965653419494629
독립 : 3.0, 종속 : 7.0, 예측 : 7.002187252044678
독립 : 4.0, 종속 : 9.0, 예측 : 9.038721084594727


In [441]:
# sklearn에서 제공하는 캘리포니아 데이터를 이용하여 회귀분석 
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np

In [442]:
# 데이터를 로드해서 독립 종속 변수 데이터 생성
data = fetch_california_housing()

X = data['data']
Y = data['target']

print("독립 변수 데이터의 크기 : ", X.shape)
print('종속 변수 데이터의 크기 : ', Y.shape)

독립 변수 데이터의 크기 :  (20640, 8)
종속 변수 데이터의 크기 :  (20640,)


In [443]:
# 데이터를 학습, 평가 데이터로 분할
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

In [444]:
# StandardScaler를 이용하여 스케일링 -> train 데이터를 기준으로 fiiting을 하고
# train, test 모두 같은 fit model 에서 변환 작업 (데이터 누수 방지)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

In [445]:
# 총 4개의 데이터의 타입은 array 형태 -> Tensor 형태로 변환
X_train_sc_tensor = torch.tensor(X_train_sc, dtype=torch.float32)
X_test_sc_tensor = torch.tensor(X_test_sc, dtype=torch.float32)
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
# Y의 데이터들은 현재 1차원 -> 2차원 변경
Y_train_tensor = torch.tensor( Y_train.reshape(-1, 1), dtype=torch.float32 )
Y_test_tensor = torch.tensor(Y_test.reshape(-1, 1), dtype=torch.float32)

In [446]:
# class 생성
class Reg(nn.Module):
    # 생성자 함수에서 self 외의 매개변수를 하나 생성 ->
    # nn.Linear()에서 사용할 첫번째 인자 값(입력 값의 열의 크기)
    def __init__(self, _dim):
        super(Reg, self).__init__()
        self.linear = nn.Linear(_dim, 1)
    
    def forward(self, x):
        return self.linear(x)

In [447]:
# 모델을 생성 -> 학습 데이터의 피쳐의 개수
reg_model = Reg( X_train.shape[1] )

In [448]:
criterion = nn.MSELoss()
optimizer = optim.SGD(reg_model.parameters(), lr = 0.01)

In [449]:
for epoch in range(300):
    # 순전파
    Y_pred = reg_model(X_train_sc_tensor)
    # 손실
    loss = criterion(Y_pred, Y_train_tensor)
    # 기울기 초기화
    optimizer.zero_grad()
    # 역전파
    loss.backward()
    # 가중치 업데이터
    optimizer.step()
    # 30회마다 loss의 변화
    if (epoch + 1) % 30 == 0:
        print(f'Epoch : {epoch+1}, Loss : {round(loss.item(), 4)}')

Epoch : 30, Loss : 2.326
Epoch : 60, Loss : 1.1722
Epoch : 90, Loss : 0.8263
Epoch : 120, Loss : 0.7137
Epoch : 150, Loss : 0.6708
Epoch : 180, Loss : 0.6496
Epoch : 210, Loss : 0.6357
Epoch : 240, Loss : 0.6248
Epoch : 270, Loss : 0.6153
Epoch : 300, Loss : 0.6069


In [450]:
# 현재 모델의 학습 모드에서 평가 모드로 변환 (학습이 중지가 아니라 모드의 변환)
# eval() : 평가 모드 전환
# train() : 학습 모드 전환
reg_model.eval()

Reg(
  (linear): Linear(in_features=8, out_features=1, bias=True)
)

In [451]:
# 예측과 평가
# torch.nograd() -> 평가시 gradient 계산을 비활성화 (속도 빨라짐, 메모리 소모 줄어듬)
with torch.no_grad():
    Y_pred = reg_model(X_test_sc_tensor)
    test_loss = criterion(Y_pred, Y_test_tensor)

In [452]:
# 성능 평가
# MSE, RMSE 지표 확인
rmse = np.sqrt(test_loss.item())
print("테스트 데이터의 MSE : ", round(test_loss.item(), 4))
print("테스트 데이터 RMSE : ", round(rmse, 4))

테스트 데이터의 MSE :  0.6203
테스트 데이터 RMSE :  0.7876


In [453]:
# 실제 값과 예측 값을 확인
for i in range(10):
    print(f"실제 값 : {Y_test[i]}, 예측 값 : {round(Y_pred[i].item(), 3)}")

실제 값 : 0.477, 예측 값 : 1.004
실제 값 : 0.458, 예측 값 : 1.561
실제 값 : 5.00001, 예측 값 : 2.376
실제 값 : 2.186, 예측 값 : 2.697
실제 값 : 2.78, 예측 값 : 2.13
실제 값 : 1.587, 예측 값 : 2.159
실제 값 : 1.982, 예측 값 : 2.713
실제 값 : 1.575, 예측 값 : 2.178
실제 값 : 3.4, 예측 값 : 2.109
실제 값 : 4.466, 예측 값 : 4.163


In [454]:
# 스탠다드 스케일링을 하지 않은 데이터를 이용하여 학습하고 예측을 출력

# 1. 모델 생성
reg_model2 = Reg(X_train.shape[1])

criterion2 = nn.MSELoss()
optimizer2 = optim.Adam(reg_model2.parameters(), lr = 0.01)

print(X_train_tensor.shape, X_test_tensor.shape)

torch.Size([16512, 8]) torch.Size([4128, 8])


In [455]:
for epoch in range(300):
    # 순전파
    Y_pred2 = reg_model2(X_train_tensor)
    # 손실
    loss2 = criterion2(Y_pred2, Y_train_tensor)
    # 기울기 초기화
    optimizer2.zero_grad()
    # 역전파
    loss2.backward()
    # 가중치 업데이터
    optimizer2.step()
    # 30회마다 loss의 변화
    if (epoch + 1) % 30 == 0:
        print(f'Epoch : {epoch+1}, Loss : {round(loss.item(), 4)}')

Epoch : 30, Loss : 0.6069
Epoch : 60, Loss : 0.6069
Epoch : 90, Loss : 0.6069
Epoch : 120, Loss : 0.6069
Epoch : 150, Loss : 0.6069
Epoch : 180, Loss : 0.6069
Epoch : 210, Loss : 0.6069
Epoch : 240, Loss : 0.6069
Epoch : 270, Loss : 0.6069
Epoch : 300, Loss : 0.6069


In [456]:
reg_model2.eval()
with torch.no_grad():
    Y_pred2 = reg_model2(X_test_tensor)
    test_loss2 = criterion2(Y_pred2, Y_test_tensor).item()

In [457]:
for i in range(10):
    print(f"실제 : {Y_test[i]}, 예측 : {round(Y_pred2[i].item(), 4)}")

실제 : 0.477, 예측 : 1.1838
실제 : 0.458, 예측 : 0.8502
실제 : 5.00001, 예측 : -2.0289
실제 : 2.186, 예측 : 0.8891
실제 : 2.78, 예측 : -1.1864
실제 : 1.587, 예측 : 3.6914
실제 : 1.982, 예측 : -3.2511
실제 : 1.575, 예측 : -2.0988
실제 : 3.4, 예측 : 1.5488
실제 : 4.466, 예측 : -4.799


In [458]:
rmse2 = np.sqrt(test_loss2)
print("스케일링 x, 테스트 MSE : ", round(test_loss2, 4))
print("스케일링 x, 테스트 RMSE : ", round(rmse2, 4))

스케일링 x, 테스트 MSE :  18.6938
스케일링 x, 테스트 RMSE :  4.3236


In [459]:
# 파이토치를 이용한 분류 분석
# iris 데이터를 이용
from sklearn.datasets import load_iris
# 분류검증 정확도
from sklearn.metrics import accuracy_score

In [460]:
iris = load_iris()
X = iris['data']
Y = iris['target']

In [461]:
# 데이터 분할
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size=0.2,
    random_state=42
)

In [462]:
# 스탠다드스케일러를 이용한 스케일링
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [463]:
# 분류 모델에서 Tensor 데이터로 변환
# 독립 변수 -> float32 타입으로 변경
# 종속 변수 -> long 타입으로 변경
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
Y_train_tensor = torch.tensor(Y_train, dtype=torch.long)
Y_test_tensor = torch.tensor(Y_test, dtype=torch.long)

In [464]:
Y_train_tensor

tensor([0, 0, 1, 0, 0, 2, 1, 0, 0, 0, 2, 1, 1, 0, 0, 1, 2, 2, 1, 2, 1, 2, 1, 0,
        2, 1, 0, 0, 0, 1, 2, 0, 0, 0, 1, 0, 1, 2, 0, 1, 2, 0, 2, 2, 1, 1, 2, 1,
        0, 1, 2, 0, 0, 1, 1, 0, 2, 0, 0, 1, 1, 2, 1, 2, 2, 1, 0, 0, 2, 2, 0, 0,
        0, 1, 2, 0, 2, 2, 0, 1, 1, 2, 1, 2, 0, 2, 1, 2, 1, 1, 1, 0, 1, 1, 0, 1,
        2, 2, 0, 1, 2, 2, 0, 2, 0, 1, 2, 2, 1, 2, 1, 1, 2, 2, 0, 1, 2, 0, 1, 2])

In [465]:
# 모델의 정의
class IrisClass(nn.Module):
    def __init__(self):
        # 부모 클래스의 생성자 함수 실행
        super(IrisClass, self).__init__()
        # 분류 모델을 선택
        self.model = nn.Sequential(
            # Linear() -> 첫번째 인자 입력 값의 열의 개수
            # 두번째 인자 값을 분류 클래스의 개수
            nn.Linear(4, 3)
        )
    
    def forward(self, x):
        return self.model(x)

In [466]:
# 모델의 생성
cls_model = IrisClass()

In [467]:
# 손실 함수 / 옵티마이저 생성
# 분류 모델의 손실 함수 
criterion = nn.CrossEntropyLoss()
# 옵티마이저
optimizer = optim.Adam(cls_model.parameters(), lr=0.01)

In [469]:
# 반복 학습 시작
for epoch in range(300):
    Y_pred = cls_model(X_train_tensor)

    loss = criterion(Y_pred, Y_train_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # 30회마다 loss를 확인
    if (epoch + 1) % 30 == 0:
        print(f"Epoch : {epoch+1}, Loss : {round(loss.item(), 4)}")

Epoch : 30, Loss : 0.8008
Epoch : 60, Loss : 0.5331
Epoch : 90, Loss : 0.4367
Epoch : 120, Loss : 0.3845
Epoch : 150, Loss : 0.3476
Epoch : 180, Loss : 0.3181
Epoch : 210, Loss : 0.2931
Epoch : 240, Loss : 0.2715
Epoch : 270, Loss : 0.2526
Epoch : 300, Loss : 0.236


In [475]:
cls_model.eval()
with torch.no_grad():
    pred = cls_model(X_test_tensor)
    _, pred_idx = torch.max(pred,1)

    # 정확도를 계산
    acc = accuracy_score(Y_test, pred_idx)

print("정확도 : ", acc)

정확도 :  1.0


In [473]:
# torch.max() -> 2차원 리스트 형태의 데이터에서 각 원소별 위치, 값을 되돌려주는 함수
torch.max(pred, 1)

torch.return_types.max(
values=tensor([2.0158, 3.5219, 6.6735, 1.5866, 2.5195, 2.9658, 1.3240, 3.2898, 2.8866,
        1.8884, 2.2354, 3.1380, 3.3953, 3.1980, 4.2625, 1.1566, 3.6202, 2.0913,
        1.5990, 3.7215, 3.4674, 1.7309, 3.3484, 3.5530, 3.5832, 3.3966, 4.0785,
        3.7713, 2.9114, 3.1446]),
indices=tensor([1, 0, 2, 1, 1, 0, 1, 2, 1, 1, 2, 0, 0, 0, 0, 1, 2, 1, 1, 2, 0, 2, 0, 2,
        2, 2, 2, 2, 0, 0]))

In [476]:
for i in range(10):
    print(f"실제 : {Y_test[i]}, 예측 : {pred_idx[i].item()}")

실제 : 1, 예측 : 1
실제 : 0, 예측 : 0
실제 : 2, 예측 : 2
실제 : 1, 예측 : 1
실제 : 1, 예측 : 1
실제 : 0, 예측 : 0
실제 : 1, 예측 : 1
실제 : 2, 예측 : 2
실제 : 1, 예측 : 1
실제 : 1, 예측 : 1


In [479]:
from sklearn.preprocessing import LabelEncoder

In [480]:
import pandas as pd
body = pd.read_csv("../data/bodyPerformance.csv")
body.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13393 entries, 0 to 13392
Data columns (total 12 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   age                      13393 non-null  float64
 1   gender                   13393 non-null  object 
 2   height_cm                13393 non-null  float64
 3   weight_kg                13393 non-null  float64
 4   body fat_%               13393 non-null  float64
 5   diastolic                13393 non-null  float64
 6   systolic                 13393 non-null  float64
 7   gripForce                13393 non-null  float64
 8   sit and bend forward_cm  13393 non-null  float64
 9   sit-ups counts           13393 non-null  float64
 10  broad jump_cm            13393 non-null  float64
 11  class                    13393 non-null  object 
dtypes: float64(10), object(2)
memory usage: 1.2+ MB


In [482]:
obj_cols = body.select_dtypes('object').columns.tolist()

for col in obj_cols:
    
    body[col] = LabelEncoder().fit_transform(body[col].values)

In [483]:
body.head()

,age,gender,height_cm,weight_kg,body fat_%,diastolic,systolic,gripForce,sit and bend forward_cm,sit-ups counts,broad jump_cm,class
0,27.0,1,172.3,75.24,21.3,80.0,130.0,54.9,18.4,60.0,217.0,2
1,25.0,1,165.0,55.80,15.7,77.0,126.0,36.4,16.3,53.0,229.0,0
2,31.0,1,179.6,78.00,20.1,92.0,152.0,44.8,12.0,49.0,181.0,2
3,32.0,1,174.5,71.10,18.4,76.0,147.0,41.4,15.2,53.0,219.0,1
4,28.0,1,173.8,67.70,17.1,70.0,127.0,43.5,27.1,45.0,217.0,1


In [484]:
X = body.drop('class', axis= 1).values
Y = body['class'].values

In [506]:
class BodyClass(nn.Module):
    def __init__(self):
        # 부모 클래스의 생성자 함수 실행
        super(BodyClass, self).__init__()
        # 분류 모델을 선택
        self.model = nn.Sequential(
            # Linear() -> 첫번째 인자 입력 값의 열의 개수
            # 두번째 인자 값을 분류 클래스의 개수
            nn.Linear(11, 4)
        )
    
    def forward(self, x):
        return self.model(x)

In [507]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
Y_train_tensor = torch.tensor(Y_train, dtype=torch.long)
Y_test_tensor = torch.tensor(Y_test, dtype=torch.long)


In [508]:
X_train_tensor.shape[1]


11

In [509]:
Y_train_tensor

tensor([3, 1, 3,  ..., 0, 3, 1])

In [510]:
cls_model = BodyClass()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(cls_model.parameters(), lr=0.01)

for epoch in range(200):
    Y_pred = cls_model(X_train_tensor)

    loss = criterion(Y_pred, Y_train_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

if (epoch + 1) % 20 == 0:
        print(f"Epoch : {epoch+1}, Loss : {round(loss.item(), 4)}")


Epoch : 200, Loss : 0.8888


In [511]:
cls_model.eval()
with torch.no_grad():
    pred = cls_model(X_test_tensor)
    _, pred_idx = torch.max(pred,1)

    # 정확도를 계산
    acc = accuracy_score(Y_test, pred_idx)

print("정확도 : ", acc)

정확도 :  0.6147816349384099
